# Kirsch 2D PINN - Google Colab Training Notebook (Full Domain)
本 Notebook 会将相关的 Python 源码直接写入 Colab 的文件系统，然后启动训练。
运行前请确保在 Colab 中启用了 GPU (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
%%writefile plot_analytical_2d.py
import numpy as np
import matplotlib.pyplot as plt
import os

def kirsch_analytical_sigma_xx(x, y, a=1.0, sigma_0=10.0):
    """
    计算 Kirsch 问题理论精确解下的 sigma_xx
    """
    r = np.sqrt(x**2 + y**2)
    theta = np.arctan2(y, x)
    
    # 避免除以 0
    r = np.clip(r, 1e-6, None)
    
    # 极坐标下的精确应力分布
    sigma_rr = (sigma_0 / 2) * (1 - (a/r)**2) + (sigma_0 / 2) * (1 - 4*(a/r)**2 + 3*(a/r)**4) * np.cos(2*theta)
    sigma_tt = (sigma_0 / 2) * (1 + (a/r)**2) - (sigma_0 / 2) * (1 + 3*(a/r)**4) * np.cos(2*theta)
    tau_rt = - (sigma_0 / 2) * (1 + 2*(a/r)**2 - 3*(a/r)**4) * np.sin(2*theta)
    
    # 坐标变换：极坐标系 -> 笛卡尔坐标系
    sigma_xx = sigma_rr * np.cos(theta)**2 + sigma_tt * np.sin(theta)**2 - 2 * tau_rt * np.sin(theta) * np.cos(theta)
    
    return sigma_xx

def kirsch_analytical_displacements(x, y, a=1.0, E=1000.0, nu=0.3, sigma_0=10.0):
    """
    计算 Kirsch 问题理论精确解下的位移 u 和 v (基于平面应力假设)
    """
    r = np.sqrt(x**2 + y**2)
    theta = np.arctan2(y, x)
    r = np.clip(r, 1e-6, None)
    
    G = E / (2 * (1 + nu))
    kappa = (3 - nu) / (1 + nu)  # 平面应力条件下的 Kolosov 常数
    
    # 极坐标系下的位移
    u_r = (sigma_0 * a / (8 * G)) * (
        (r / a) * (kappa - 1) + 2 * (a / r) + np.cos(2 * theta) * (2 * (r / a) + (a / r) * (kappa + 1) - (a / r)**3)
    )
    
    u_theta = (sigma_0 * a / (8 * G)) * np.sin(2 * theta) * (
        -2 * (r / a) - (a / r) * (kappa - 1) - (a / r)**3
    )
    
    # 极坐标位移变换到笛卡尔坐标系
    u = u_r * np.cos(theta) - u_theta * np.sin(theta)
    v = u_r * np.sin(theta) + u_theta * np.cos(theta)
    
    return u, v

def plot_analytical():
    W, H, a = 10.0, 10.0, 1.0
    sigma_0 = 10.0
    E, nu = 1000.0, 0.3
    
    # 创建完整的正方形网格 (-W 到 W, -H 到 H)
    grid_res = 300
    x_list = np.linspace(-W, W, grid_res)
    y_list = np.linspace(-H, H, grid_res)
    X, Y = np.meshgrid(x_list, y_list)
    
    # 过滤孔洞区域
    mask = (X**2 + Y**2) >= a**2
    X_valid = X[mask]
    Y_valid = Y[mask]
    
    # 计算精确解
    sigma_xx_valid = kirsch_analytical_sigma_xx(X_valid, Y_valid, a=a, sigma_0=sigma_0)
    u_exact, v_exact = kirsch_analytical_displacements(X_valid, Y_valid, a=a, E=E, nu=nu, sigma_0=sigma_0)
    
    def fill_grid(data):
        grid = np.full_like(X, np.nan)
        grid[mask] = data
        return grid
        
    S_xx = fill_grid(sigma_xx_valid)
    U_grid = fill_grid(u_exact)
    V_grid = fill_grid(v_exact)
    
    # 开始绘图
    fig, axes = plt.subplots(1, 3, figsize=(24, 7))
    
    fields = [S_xx, U_grid, V_grid]
    titles = [r'Stress $\sigma_{xx}$', r'Displacement $u$ (x-dir)', r'Displacement $v$ (y-dir)']
    cmaps = ['jet', 'coolwarm', 'coolwarm']
    
    # 画出完整的圆孔边界线
    theta_line = np.linspace(0, 2*np.pi, 200)
    hole_x = a * np.cos(theta_line)
    hole_y = a * np.sin(theta_line)
    
    for i in range(3):
        ax = axes[i]
        contour = ax.contourf(X, Y, fields[i], levels=60, cmap=cmaps[i])
        ax.plot(hole_x, hole_y, 'k-', linewidth=3)
        ax.set_aspect('equal', adjustable='box')
        ax.set_title(f'Kirsch 2D Exact: {titles[i]} (Full Plate)', fontsize=14, fontweight='bold')
        ax.set_xlabel('x', fontsize=12)
        ax.set_ylabel('y', fontsize=12)
        plt.colorbar(contour, ax=ax)
    
    save_path = os.path.join(os.path.dirname(__file__), 'result_2d_kirsch_exact_full_all.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Exact analytical chart saved to {save_path}")

if __name__ == '__main__':
    plot_analytical()


In [ ]:
%%writefile geometry.py
import torch
import numpy as np


class KirschGeometrySampler:
    """
    二维 Kirsch 问题（带孔平板单向拉伸）的几何采样器
    计算域为全域: [-W, W] x [-H, H] 且 x^2 + y^2 >= a^2
    """

    def __init__(self, W: float, H: float, a: float, device='cpu'):
        self.W = W
        self.H = H
        self.a = a
        self.a2 = a * a
        self.device = device

    def _collect_points(self, n_target: int, x_gen, y_gen, mask_fn, oversample: float = 2.0):
        """在 device 上矢量采样，避免 CPU while 循环"""
        chunks_x, chunks_y = [], []
        collected = 0
        while collected < n_target:
            batch = max(int(n_target * oversample), n_target - collected + 64)
            x = x_gen(batch)
            y = y_gen(batch)
            mask = mask_fn(x, y)
            vx, vy = x[mask], y[mask]
            if vx.numel() == 0:
                continue
            chunks_x.append(vx)
            chunks_y.append(vy)
            collected += vx.numel()

        x_out = torch.cat(chunks_x, dim=0)[:n_target].reshape(-1, 1)
        y_out = torch.cat(chunks_y, dim=0)[:n_target].reshape(-1, 1)
        return x_out, y_out

    def sample_domain(self, n_f: int, requires_grad: bool = True):
        def x_gen(n):
            return (torch.rand(n, 1, device=self.device) * 2.0 - 1.0) * self.W

        def y_gen(n):
            return (torch.rand(n, 1, device=self.device) * 2.0 - 1.0) * self.H

        def mask_fn(x, y):
            return (x * x + y * y) >= self.a2

        x_f, y_f = self._collect_points(n_f, x_gen, y_gen, mask_fn)
        if requires_grad:
            x_f = x_f.requires_grad_(True)
            y_f = y_f.requires_grad_(True)
        return x_f, y_f

    def sample_boundaries(self, n_bc: int, requires_grad: bool = True):
        y_left = (torch.rand(n_bc, 1, device=self.device) * 2.0 - 1.0) * self.H
        x_left = torch.full_like(y_left, -self.W)

        x_bottom = (torch.rand(n_bc, 1, device=self.device) * 2.0 - 1.0) * self.W
        y_bottom = torch.full_like(x_bottom, -self.H)

        y_right = (torch.rand(n_bc, 1, device=self.device) * 2.0 - 1.0) * self.H
        x_right = torch.full_like(y_right, self.W)

        x_top = (torch.rand(n_bc, 1, device=self.device) * 2.0 - 1.0) * self.W
        y_top = torch.full_like(x_top, self.H)

        theta = torch.rand(n_bc, 1, device=self.device) * (2.0 * np.pi)
        x_hole = self.a * torch.cos(theta)
        y_hole = self.a * torch.sin(theta)

        def maybe_grad(t):
            return t.requires_grad_(True) if requires_grad else t

        bc_dict = {
            'left': (maybe_grad(x_left), maybe_grad(y_left)),
            'bottom': (maybe_grad(x_bottom), maybe_grad(y_bottom)),
            'right': (maybe_grad(x_right), maybe_grad(y_right)),
            'top': (maybe_grad(x_top), maybe_grad(y_top)),
            'hole': (maybe_grad(x_hole), maybe_grad(y_hole)),
        }
        return bc_dict

    def sample_dense_near_hole(self, n_dense: int, max_radius: float = 3.0, requires_grad: bool = True):
        max_r2 = max_radius * max_radius

        def x_gen(n):
            return (torch.rand(n, 1, device=self.device) * 2.0 - 1.0) * max_radius

        def y_gen(n):
            return (torch.rand(n, 1, device=self.device) * 2.0 - 1.0) * max_radius

        def mask_fn(x, y):
            r2 = x * x + y * y
            return (r2 >= self.a2) & (r2 <= max_r2)

        x_dense, y_dense = self._collect_points(n_dense, x_gen, y_gen, mask_fn, oversample=2.5)
        if requires_grad:
            x_dense = x_dense.requires_grad_(True)
            y_dense = y_dense.requires_grad_(True)
        return x_dense, y_dense

    def sample_radial_biased(self, n: int, k: float = 2.0, requires_grad: bool = True):
        """
        径向偏置采样: p(x,y) ∝ (a/r)^k
        - k=0: 等效均匀采样
        - k=2: 与 Kirsch 应力场衰减阶次匹配（高效）
        - k=3+: 更极端地集中于孔口
        
        原理：先在全域均匀提案，再用 (a/r)^k 做接受-拒绝过滤。
        在 r=a 处接受率=1（最高），r=2a 处接受率=(1/2)^k，以此类推。
        """
        def x_gen(m):
            return (torch.rand(m, 1, device=self.device) * 2.0 - 1.0) * self.W

        def y_gen(m):
            return (torch.rand(m, 1, device=self.device) * 2.0 - 1.0) * self.H

        def mask_fn(x, y):
            r2 = x * x + y * y
            outside_hole = r2 >= self.a2
            # r clamped to avoid division by zero for points exactly at origin
            r = torch.sqrt(r2.clamp(min=self.a2))
            # Acceptance probability: 1 at r=a, decays as (a/r)^k outward
            p_accept = (self.a / r) ** k
            accept = torch.rand_like(p_accept) < p_accept
            return outside_hole & accept

        # k>0 reduces effective density, so we oversample more aggressively
        # Expected acceptance rate ≈ (a/(avg_r))^k; use large oversample factor
        oversample = max(4.0, 2.0 * (self.W / self.a) ** (k * 0.5))
        x_out, y_out = self._collect_points(n, x_gen, y_gen, mask_fn, oversample=oversample)
        if requires_grad:
            x_out = x_out.requires_grad_(True)
            y_out = y_out.requires_grad_(True)
        return x_out, y_out


In [ ]:
%%writefile network.py
import torch
import torch.nn as nn


class PINN_2D_MLP(nn.Module):
    """
    二维神经网络模型 (MIMO)

    【关键设计】直接以极坐标为主要输入特征，去掉 Fourier 特征以消除数值干扰。
    Kirsch 问题的精确解结构:
      u_r = f(r/a) * cos(2θ),  u_θ = g(r/a) * sin(2θ)
      σ_rr = h(a/r) * cos(2θ), σ_θθ = k(a/r) * cos(2θ)

    输入特征 (5维):
      [x/W, y/W, log(r/a), cos(2θ), sin(2θ)]
    其中:
      - x/W, y/W: 归一化坐标，帮助学习线性远场位移
      - log(r/a): 捕捉 1/r^2 衰减（取对数后变为线性，易于学习）
      - cos(2θ), sin(2θ): Kirsch 角向依赖的自然基底，消除频谱偏差
    """

    def __init__(
        self,
        in_dim=2,
        out_dim=2,
        hidden_layers=6,
        hidden_neurons=96,
        a: float = 1.0,
        W: float = 10.0,
    ):
        super().__init__()
        self.a = a
        self.W = W

        # 输入维度: x/W, y/W, log(r/a), cos2θ, sin2θ = 5
        input_dim = 5
        layers = []
        layers.append(nn.Linear(input_dim, hidden_neurons))
        layers.append(nn.SiLU())

        for _ in range(hidden_layers):
            layers.append(nn.Linear(hidden_neurons, hidden_neurons))
            layers.append(nn.SiLU())

        layers.append(nn.Linear(hidden_neurons, out_dim))
        self.net = nn.Sequential(*layers)

        # 初始化最后一层为接近零，让训练从小扰动出发
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def _features(self, x, y):
        """计算所有输入特征"""
        xn = x / self.W
        yn = y / self.W
        r = torch.sqrt(x ** 2 + y ** 2).clamp(min=self.a * 1e-3)
        log_r = torch.log(r / self.a)       # 范围 [0, log(W/a)] ≈ [0, 2.3]
        r2 = r ** 2
        cos2t = (x ** 2 - y ** 2) / r2     # cos(2θ) = (x²-y²)/r²
        sin2t = 2.0 * x * y / r2           # sin(2θ) = 2xy/r²
        return torch.cat([xn, yn, log_r, cos2t, sin2t], dim=1)

    def forward(self, x, y):
        feat = self._features(x, y)
        out = self.net(feat)
        N_u = out[:, 0:1]
        N_v = out[:, 1:2]

        # 全域问题不再使用对称 Ansatz
        u_pred = N_u
        v_pred = N_v

        return u_pred, v_pred


In [ ]:
%%writefile evaluator.py
import torch
import torch.nn as nn


class KirschPhysicsEvaluator(nn.Module):
    """
    二维 Kirsch 问题物理引擎与张量微分计算器
    """

    def __init__(
        self,
        model,
        E: float,
        nu: float,
        sigma_0: float,
        w_pde: float = 10.0,
        w_far: float = 30.0,    # 远场权重
        w_hole: float = 500.0,  # 孔口无牵引力权重
        w_rbm: float = 1e4,     # 刚体位移约束权重，必须极大约束以锚定平板
    ):
        super().__init__()
        self.model = model
        self.E = E
        self.nu = nu
        self.sigma_0 = sigma_0
        self.w_pde = w_pde
        self.w_far = w_far
        self.w_hole = w_hole
        self.w_rbm = w_rbm

        self.G = E / (2 * (1 + nu))
        self.C11 = E / (1 - nu**2)
        self.C12 = E * nu / (1 - nu**2)

    def compute_stresses(self, x, y):
        u, v = self.model(x, y)

        du = torch.autograd.grad(u, [x, y], grad_outputs=torch.ones_like(u), create_graph=True)
        du_dx, du_dy = du[0], du[1]

        dv = torch.autograd.grad(v, [x, y], grad_outputs=torch.ones_like(v), create_graph=True)
        dv_dx, dv_dy = dv[0], dv[1]

        sigma_xx = self.C11 * du_dx + self.C12 * dv_dy
        sigma_yy = self.C12 * du_dx + self.C11 * dv_dy
        tau_xy = self.G * (du_dy + dv_dx)

        return sigma_xx, sigma_yy, tau_xy, u, v

    def pde_residual(self, x_f, y_f):
        sigma_xx, sigma_yy, tau_xy, _, _ = self.compute_stresses(x_f, y_f)

        dsxx_dx = torch.autograd.grad(sigma_xx, x_f, grad_outputs=torch.ones_like(sigma_xx), create_graph=True)[0]
        dtxy_dy = torch.autograd.grad(tau_xy, y_f, grad_outputs=torch.ones_like(tau_xy), create_graph=True)[0]
        dtxy_dx = torch.autograd.grad(tau_xy, x_f, grad_outputs=torch.ones_like(tau_xy), create_graph=True)[0]
        dsyy_dy = torch.autograd.grad(sigma_yy, y_f, grad_outputs=torch.ones_like(sigma_yy), create_graph=True)[0]

        res_x = dsxx_dx + dtxy_dy
        res_y = dtxy_dx + dsyy_dy

        return (res_x / self.sigma_0) ** 2 + (res_y / self.sigma_0) ** 2

    def pde_loss(self, x_f, y_f):
        return torch.mean(self.pde_residual(x_f, y_f))

    def bc_losses(self, bc_dict, a):
        """五条边界合并为一次前向，减少重复 autograd 图构建"""
        keys = ('left', 'bottom', 'right', 'top', 'hole')
        xs = torch.cat([bc_dict[k][0] for k in keys], dim=0)
        ys = torch.cat([bc_dict[k][1] for k in keys], dim=0)
        sigma_xx, sigma_yy, tau_xy, _, _ = self.compute_stresses(xs, ys)

        s = self.sigma_0
        off = 0
        far_terms = []
        hole_terms = []

        for key in keys:
            n = bc_dict[key][0].shape[0]
            sl = slice(off, off + n)
            off += n
            sxx, syy, txy = sigma_xx[sl], sigma_yy[sl], tau_xy[sl]

            if key in ('left', 'right'):
                # 左右都是受拉远场
                far_terms.append(((sxx - s) / s) ** 2)
                far_terms.append((txy / s) ** 2)
            elif key in ('top', 'bottom'):
                # 上下都是自由远场
                far_terms.append((syy / s) ** 2)
                far_terms.append((txy / s) ** 2)
            else:
                x_hole, y_hole = bc_dict[key]
                nx = -x_hole / a
                ny = -y_hole / a
                Tx = sxx * nx + txy * ny
                Ty = txy * nx + syy * ny
                hole_terms.append((Tx / s) ** 2)
                hole_terms.append((Ty / s) ** 2)

        loss_far = torch.mean(torch.cat(far_terms)) if far_terms else torch.tensor(0.0, device=xs.device)
        loss_hole = torch.mean(torch.cat(hole_terms)) if hole_terms else torch.tensor(0.0, device=xs.device)
        return loss_far, loss_hole

    def compute_total_loss(self, x_f, y_f, bc_dict, a, W=10.0, H=10.0, sync_metrics=False):
        loss_pde = self.pde_loss(x_f, y_f)
        loss_far, loss_hole = self.bc_losses(bc_dict, a)

        # 刚体位移约束 (Rigid Body Motion)
        # 固定四个对称轴端点: u(0, H)=0, u(0, -H)=0, v(W, 0)=0, v(-W, 0)=0
        x_rbm = torch.tensor([[0.0], [0.0], [W], [-W]], device=x_f.device)
        y_rbm = torch.tensor([[H], [-H], [0.0], [0.0]], device=x_f.device)
        u_rbm, v_rbm = self.model(x_rbm, y_rbm)
        
        # 为了与无量纲的应力损失匹配，除以特征长度和应变尺度
        # 或者直接作为绝对值的 MSE。由于位移较小，加个大权重 w_rbm
        loss_rbm = torch.mean(u_rbm[0:2]**2) + torch.mean(v_rbm[2:4]**2)

        total_loss = (
            self.w_pde * loss_pde
            + self.w_far * loss_far
            + self.w_hole * loss_hole
            + self.w_rbm * loss_rbm
        )

        raw = {
            'pde': loss_pde,
            'far': loss_far,
            'hole': loss_hole,
            'rbm': loss_rbm,
            'total': total_loss,
        }
        if sync_metrics:
            raw = {k: v.item() for k, v in raw.items()}
        return total_loss, raw


In [ ]:
%%writefile main.py
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
from geometry import KirschGeometrySampler
from network import PINN_2D_MLP
from evaluator import KirschPhysicsEvaluator
from plot_analytical_2d import kirsch_analytical_sigma_xx

# --- 训练加速配置 ---
TRAIN_CONFIG = {
    # 配点池 - 径向偏置采样 p(r) ∝ (a/r)^k，取代两阶段 pool+dense 方案
    'pool_size': 30000,            # 总配点数（原 pool_size + dense_pool_size）
    'biased_k': 2.0,               # 偏置指数：2.0 匹配 Kirsch 应力衰减阶次
    'max_pool_size': 50000,
    'mini_batch_size': 8192,
    'bc_points': 800,
    # Adam
    'adam_steps': 8000,
    'adam_lr': 5e-4,
    # 日志 / 评估频率
    'log_interval': 50,
    'eval_interval': 200,
    'bc_resample_interval': 300,
    'rar_interval': 1500,
    'rar_probe': 8000,
    'rar_add': 1000,
    # L-BFGS 精修
    'use_lbfgs': True,
    'lbfgs_steps': 500,
    # torch.compile 可加速前向；若报错改为 False
    'use_compile': False,
}


def setup_device():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type == 'cuda':
        torch.backends.cudnn.benchmark = True
        torch.set_float32_matmul_precision('high')
    return device


def maybe_compile(model, enabled: bool):
    if not enabled or not hasattr(torch, 'compile'):
        return model
    try:
        return torch.compile(model)
    except Exception as exc:
        print(f"torch.compile skipped: {exc}")
        return model


def build_domain_pool(sampler, cfg):
    """构建径向偏置采样池，p(r) ∝ (a/r)^k 自然地在孔口聚集更多配点"""
    return sampler.sample_radial_biased(
        cfg['pool_size'], k=cfg.get('biased_k', 2.0), requires_grad=False
    )


def sample_mini_batch(x_pool, y_pool, batch_size):
    """从径向偏置池中随机抽取 mini-batch，池本身已编码密度信息"""
    idx = torch.randint(0, x_pool.shape[0], (batch_size,), device=x_pool.device)
    x = x_pool[idx].clone().requires_grad_(True)
    y = y_pool[idx].clone().requires_grad_(True)
    return x, y


def rar_refine_pool(sampler, evaluator, x_pool, y_pool, cfg):
    """RAR: 探测点也使用径向偏置采样，在孔边高残差区更容易采到新点"""
    x_probe, y_probe = sampler.sample_radial_biased(
        cfg['rar_probe'], k=cfg.get('biased_k', 2.0), requires_grad=True
    )
    residuals = evaluator.pde_residual(x_probe, y_probe).detach().squeeze()
    _, top_idx = torch.topk(residuals, min(cfg['rar_add'], residuals.numel()))

    x_new = x_probe[top_idx].detach()
    y_new = y_probe[top_idx].detach()
    x_pool = torch.cat([x_pool, x_new], dim=0)
    y_pool = torch.cat([y_pool, y_new], dim=0)

    max_n = cfg['max_pool_size']
    if x_pool.shape[0] > max_n:
        keep = torch.randperm(x_pool.shape[0], device=x_pool.device)[:max_n]
        x_pool = x_pool[keep]
        y_pool = y_pool[keep]
    return x_pool, y_pool


def compute_sigma_xx_relative_l2_fast(evaluator, x_eval, y_eval, exact_np):
    """训练监控用：解析解预计算，仅对网络求导一次"""
    x_eval = x_eval.clone().requires_grad_(True)
    y_eval = y_eval.clone().requires_grad_(True)
    sigma_xx, _, _, _, _ = evaluator.compute_stresses(x_eval, y_eval)
    pred = sigma_xx.detach().cpu().numpy().flatten()
    rel_l2 = np.linalg.norm(pred - exact_np) / np.linalg.norm(exact_np)
    return rel_l2


def compute_sigma_xx_relative_l2(evaluator, X_valid, Y_valid, a, sigma_0):
    device = next(evaluator.model.parameters()).device
    x_test = torch.tensor(X_valid, dtype=torch.float32, device=device).view(-1, 1).requires_grad_(True)
    y_test = torch.tensor(Y_valid, dtype=torch.float32, device=device).view(-1, 1).requires_grad_(True)

    sigma_xx, _, _, _, _ = evaluator.compute_stresses(x_test, y_test)
    pred = sigma_xx.detach().cpu().numpy().flatten()
    exact = kirsch_analytical_sigma_xx(X_valid, Y_valid, a=a, sigma_0=sigma_0)
    rel_l2 = np.linalg.norm(pred - exact) / np.linalg.norm(exact)
    return rel_l2, pred, exact


def plot_stress_comparison(X, Y, mask, pred, exact, a, save_path):
    S_pred = np.full_like(X, np.nan)
    S_exact = np.full_like(X, np.nan)
    S_err = np.full_like(X, np.nan)

    S_pred[mask] = pred
    S_exact[mask] = exact
    S_err[mask] = np.abs(pred - exact)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    titles = [
        'Predicted $\\sigma_{xx}$',
        'Analytical $\\sigma_{xx}$',
        'Absolute Error $|\\sigma_{xx}^{pred} - \\sigma_{xx}^{exact}|$',
    ]
    datas = [S_pred, S_exact, S_err]
    cmaps = ['jet', 'jet', 'hot']

    theta = np.linspace(0, np.pi / 2, 100)
    hole_x = a * np.cos(theta)
    hole_y = a * np.sin(theta)

    for ax, data, title, cmap in zip(axes, datas, titles, cmaps):
        contour = ax.contourf(X, Y, data, levels=60, cmap=cmap)
        ax.plot(hole_x, hole_y, 'k-', linewidth=2)
        ax.set_aspect('equal', adjustable='box')
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        plt.colorbar(contour, ax=ax)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


def main():
    cfg = TRAIN_CONFIG
    device = setup_device()
    print(f"Using device: {device}")

    W, H, a = 10.0, 10.0, 1.0
    E, nu, sigma_0 = 1000.0, 0.3, 10.0

    print("1. Initializing PINN modules...")
    sampler = KirschGeometrySampler(W=W, H=H, a=a, device=device)
    x_pool, y_pool = build_domain_pool(sampler, cfg)
    bc_dict = sampler.sample_boundaries(cfg['bc_points'])

    model = PINN_2D_MLP(
        in_dim=2, out_dim=2, hidden_layers=6, hidden_neurons=96, a=a, W=W
    ).to(device)
    model = maybe_compile(model, cfg['use_compile'])
    evaluator = KirschPhysicsEvaluator(
        model, E=E, nu=nu, sigma_0=sigma_0,
        w_pde=50.0,    # 大幅提高 PDE 权重，让物理约束在孔边附近主导
        w_far=30.0,
        w_hole=50.0,   # 大幅降低（原500），避免孔边应力被强行拉至零
        w_rbm=1e4,
    ).to(device)

    writer = SummaryWriter(log_dir='runs/kirsch_2d_pinn_v3')
    optimizer_adam = torch.optim.Adam(model.parameters(), lr=cfg['adam_lr'])
    # Cosine Annealing 学习率调度：让 LR 平滑衰减到 1e-6，避免后期震荡
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_adam, T_max=cfg['adam_steps'], eta_min=1e-6
    )

    grid_res = 150
    x_list = np.linspace(-W, W, grid_res)
    y_list = np.linspace(-H, H, grid_res)
    X, Y = np.meshgrid(x_list, y_list)
    mask = (X ** 2 + Y ** 2) >= a ** 2
    X_valid = X[mask]
    Y_valid = Y[mask]

    eval_res = 64
    x_eval = np.linspace(-W, W, eval_res)
    y_eval = np.linspace(-H, H, eval_res)
    X_eval, Y_eval = np.meshgrid(x_eval, y_eval)
    mask_eval = (X_eval ** 2 + Y_eval ** 2) >= a ** 2
    X_eval_valid = X_eval[mask_eval]
    Y_eval_valid = Y_eval[mask_eval]
    exact_eval_np = kirsch_analytical_sigma_xx(X_eval_valid, Y_eval_valid, a=a, sigma_0=sigma_0)
    x_eval_t = torch.tensor(X_eval_valid, dtype=torch.float32, device=device).view(-1, 1)
    y_eval_t = torch.tensor(Y_eval_valid, dtype=torch.float32, device=device).view(-1, 1)

    print(
        f"   Pool size={x_pool.shape[0]} (radial-biased k={cfg['biased_k']}), "
        f"mini-batch={cfg['mini_batch_size']}, BC points={cfg['bc_points']}"
    )

    max_steps_adam = cfg['adam_steps']
    print("\n2. Starting Phase 1: Adam optimization (mini-batch)...")
    pbar_adam = tqdm(range(max_steps_adam), desc="Adam Phase")
    for step in pbar_adam:
        if step > 0 and step % cfg['rar_interval'] == 0:
            x_pool, y_pool = rar_refine_pool(sampler, evaluator, x_pool, y_pool, cfg)

        if step % cfg['bc_resample_interval'] == 0:
            bc_dict = sampler.sample_boundaries(cfg['bc_points'])

        x_f, y_f = sample_mini_batch(x_pool, y_pool, cfg['mini_batch_size'])

        optimizer_adam.zero_grad(set_to_none=True)
        loss, raw = evaluator.compute_total_loss(x_f, y_f, bc_dict, a=a, W=W, H=H, sync_metrics=False)
        loss.backward()
        # 梯度裁剪：防止极坐标特征在 r→a 时引发梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_adam.step()
        scheduler.step()

        if step % cfg['log_interval'] == 0:
            raw = {k: v.item() for k, v in raw.items()}
            desc = f"Loss={raw['total']:.2e}, PDE={raw['pde']:.2e}, Hole={raw['hole']:.2e}, RBM={raw['rbm']:.2e}"
            pbar_adam.set_postfix_str(desc)
            
            writer.add_scalar('Loss/Total', raw['total'], step)
            writer.add_scalar('Loss/PDE', raw['pde'], step)
            writer.add_scalar('Loss/FarField', raw['far'], step)
            writer.add_scalar('Loss/Hole', raw['hole'], step)
            writer.add_scalar('Loss/RBM', raw['rbm'], step)
            writer.add_scalar('Train/Num_Collocation', x_pool.shape[0], step)
            writer.add_scalar('Train/LR', scheduler.get_last_lr()[0], step)

    if cfg['use_lbfgs']:
        print("\n3. Starting Phase 2: L-BFGS optimization (full batch)...")
        x_f, y_f = sample_mini_batch(
            x_pool, y_pool,
            min(cfg['mini_batch_size'] * 2, x_pool.shape[0])
        )
        max_iter_lbfgs = cfg['lbfgs_steps']
        optimizer_lbfgs = torch.optim.LBFGS(
            model.parameters(),
            lr=1.0,
            max_iter=max_iter_lbfgs,
            tolerance_grad=1e-7,
            tolerance_change=1e-9,
            history_size=50,
            line_search_fn='strong_wolfe',
        )

        step_lbfgs = 0
        pbar_lbfgs = tqdm(total=max_iter_lbfgs, desc="L-BFGS Phase")

        def closure():
            nonlocal step_lbfgs
            optimizer_lbfgs.zero_grad()
            loss, raw = evaluator.compute_total_loss(x_f, y_f, bc_dict, a=a, W=W, H=H, sync_metrics=False)
            loss.backward()
            if step_lbfgs < max_iter_lbfgs and step_lbfgs % 20 == 0:
                metrics = {k: v.item() for k, v in raw.items()}
                pbar_lbfgs.set_postfix({'Loss': f"{metrics['total']:.2e}", 'Hole': f"{metrics['hole']:.2e}", 'RBM': f"{metrics['rbm']:.2e}"})
                writer.add_scalar('L-BFGS/Total_Loss', metrics['total'], step_lbfgs)
                writer.add_scalar('L-BFGS/Loss_Hole', metrics['hole'], step_lbfgs)
                writer.add_scalar('L-BFGS/Loss_RBM', metrics['rbm'], step_lbfgs)
            if step_lbfgs < max_iter_lbfgs:
                pbar_lbfgs.update(1)
            step_lbfgs += 1
            return loss

        optimizer_lbfgs.step(closure)
        pbar_lbfgs.close()
    else:
        print("\n3. Skipping L-BFGS (use_lbfgs=False).")

    writer.close()

    print("\n4. Training completed. Evaluating and plotting sigma_xx stress field...")
    model.eval()

    rel_l2, pred, exact = compute_sigma_xx_relative_l2(
        evaluator, X_valid, Y_valid, a, sigma_0
    )
    print(f"Final relative L2 error (sigma_xx): {rel_l2:.4f}")

    save_dir = os.path.dirname(__file__)
    compare_path = os.path.join(save_dir, 'result_2d_kirsch_comparison.png')
    plot_stress_comparison(X, Y, mask, pred, exact, a, compare_path)
    print(f"Comparison chart saved to {compare_path}")

    save_path = os.path.join(save_dir, 'result_2d_kirsch.png')
    S_xx = np.full_like(X, np.nan)
    S_xx[mask] = pred
    plt.figure(figsize=(10, 8))
    contour = plt.contourf(X, Y, S_xx, levels=60, cmap='jet')
    cbar = plt.colorbar(contour)
    cbar.set_label('Stress $\\sigma_{xx}$', fontsize=12)
    theta = np.linspace(0, np.pi / 2, 100)
    plt.plot(a * np.cos(theta), a * np.sin(theta), 'k-', linewidth=3)
    plt.title(f'Kirsch 2D: Predicted $\\sigma_{{xx}}$ (Rel L2 = {rel_l2:.3f})', fontsize=14)
    plt.xlabel('x', fontsize=12)
    plt.ylabel('y', fontsize=12)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Result chart saved to {save_path}")

    # 保存模型权重
    model_save_path = os.path.join(save_dir, 'kirsch_2d_model.pth')
    torch.save(model.state_dict(), model_save_path)
    print(f"Model saved to {model_save_path}")
if __name__ == '__main__':
    main()


In [ ]:
%%writefile plot_all_fields_2d.py
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
from network import PINN_2D_MLP
from evaluator import KirschPhysicsEvaluator

def plot_all_fields():
    # 0. 设置计算设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # 1. 物理与几何参数
    W, H, a = 10.0, 10.0, 1.0
    E, nu, sigma_0 = 1000.0, 0.3, 10.0

    # 2. 实例化网络并加载权重
    model = PINN_2D_MLP(
        in_dim=2, out_dim=2, hidden_layers=6, hidden_neurons=96, a=a, W=W
    ).to(device)
    
    save_dir = os.path.dirname(__file__)
    model_path = os.path.join(save_dir, 'kirsch_2d_model.pth')
    
    if not os.path.exists(model_path):
        print(f"Error: Model weights not found at {model_path}. Please train the model first.")
        return
        
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    # 实例化 evaluator 来计算应力
    evaluator = KirschPhysicsEvaluator(model, E=E, nu=nu, sigma_0=sigma_0).to(device)

    # 3. 创建网格
    grid_res = 150
    x_list = np.linspace(-W, W, grid_res)
    y_list = np.linspace(-H, H, grid_res)
    X, Y = np.meshgrid(x_list, y_list)
    
    # 过滤孔洞
    mask = (X ** 2 + Y ** 2) >= a ** 2
    X_valid = X[mask]
    Y_valid = Y[mask]

    # 转为张量
    x_test = torch.tensor(X_valid, dtype=torch.float32, device=device).view(-1, 1).requires_grad_(True)
    y_test = torch.tensor(Y_valid, dtype=torch.float32, device=device).view(-1, 1).requires_grad_(True)

    # 4. 前向计算所有场
    sigma_xx, sigma_yy, tau_xy, u, v = evaluator.compute_stresses(x_test, y_test)

    # 转回 numpy
    sxx_np = sigma_xx.detach().cpu().numpy().flatten()
    syy_np = sigma_yy.detach().cpu().numpy().flatten()
    txy_np = tau_xy.detach().cpu().numpy().flatten()
    u_np = u.detach().cpu().numpy().flatten()
    v_np = v.detach().cpu().numpy().flatten()

    # 填充回 2D 网格
    def fill_grid(data):
        grid = np.full_like(X, np.nan)
        grid[mask] = data
        return grid

    S_xx = fill_grid(sxx_np)
    S_yy = fill_grid(syy_np)
    T_xy = fill_grid(txy_np)
    U_grid = fill_grid(u_np)
    V_grid = fill_grid(v_np)

    # 5. 绘图
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    fields = [S_xx, S_yy, T_xy, U_grid, V_grid]
    titles = [
        r'Stress $\sigma_{xx}$',
        r'Stress $\sigma_{yy}$',
        r'Shear Stress $\tau_{xy}$',
        r'Displacement $u$ (x-dir)',
        r'Displacement $v$ (y-dir)'
    ]
    cmaps = ['jet', 'jet', 'jet', 'coolwarm', 'coolwarm']

    theta = np.linspace(0, 2.0 * np.pi, 200)
    hole_x = a * np.cos(theta)
    hole_y = a * np.sin(theta)

    for i in range(5):
        ax = axes[i]
        contour = ax.contourf(X, Y, fields[i], levels=60, cmap=cmaps[i])
        ax.plot(hole_x, hole_y, 'k-', linewidth=3)
        ax.set_aspect('equal', adjustable='box')
        ax.set_title(titles[i], fontsize=14)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        plt.colorbar(contour, ax=ax)

    # 隐藏第 6 个子图
    axes[5].axis('off')

    plt.tight_layout()
    out_path = os.path.join(save_dir, 'result_2d_kirsch_all_fields.png')
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"All fields chart saved to {out_path}")

if __name__ == '__main__':
    plot_all_fields()


In [ ]:
!python main.py

In [ ]:
from IPython.display import Image, display
display(Image('result_2d_kirsch_comparison.png'))
display(Image('result_2d_kirsch.png'))

In [ ]:
!python plot_all_fields_2d.py

In [ ]:
display(Image('result_2d_kirsch_all_fields.png'))

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/